# Level 3 — 불균형 대응 및 고급 Augmentation

**목표**: 다수 클래스의 정확도를 크게 희생하지 않으면서, 소수 클래스 (foggy / snowy / dawn-dusk) 의 성능을 끌어올립니다.

다음 축에서 **최소 2가지 이상** 의 기법을 적용하세요.
- Loss-level: Weighted CE, Focal Loss, LDAM, Class-Balanced Loss
- Sampling-level: class-balanced sampler
- Augmentation-level: RandAugment, Mixup, CutMix

Level 1 / 2 에서 가장 좋았던 백본을 base 로 사용하세요. wandb 를 사용하면 여러 기법의 비교 Run 을 같은 프로젝트에 모아 볼 수 있어 편리합니다.

In [2]:
import os
import sys

repo_url  = "https://github.com/Seongha-parkk/2026-HYU-AUE8088-PA2.git"
repo_name = "2026-HYU-AUE8088-PA2"

if not os.path.exists(f"/content/{repo_name}"):
    !git clone {repo_url}
else:
    # 런타임이 살아있고 이미 클론된 경우 최신 코드로 업데이트
    !git -C /content/{repo_name} pull

# 2. 작업 디렉토리를 레포지토리의 최상단(Root)으로 변경
%cd /content/{repo_name}

%load_ext autoreload
%autoreload 2

# 의존성 설치 (이미 설치된 패키지는 빠르게 skip)
!pip install -q -r requirements.txt

Cloning into '2026-HYU-AUE8088-PA2'...
remote: Enumerating objects: 172, done.
remote: Counting objects: 100% (172/172), done.
remote: Compressing objects: 100% (122/122), done.
remote: Total 172 (delta 97), reused 122 (delta 47), pack-reused 0 (from 0)
Receiving objects: 100% (172/172), 989.42 KiB | 17.67 MiB/s, done.
Resolving deltas: 100% (97/97), done.
/content/2026-HYU-AUE8088-PA2
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 274.9/274.9 kB 14.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 455.2/455.2 kB 51.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 26.4/26.4 MB 91.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 70.6 MB/s eta 0:00:00:00:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 46.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 68.4 MB/s eta 0:00:00
   ━━━━━

In [3]:
from google.colab import drive
import os

drive.mount('/drive')

DRIVE_CKPT = "/drive/MyDrive/aue8088-pa2/checkpoints"
os.makedirs(DRIVE_CKPT, exist_ok=True)

LOCAL_CKPT = os.path.abspath("../checkpoints")
if os.path.islink(LOCAL_CKPT):
    print(f"symlink 이미 존재: {LOCAL_CKPT} → {os.readlink(LOCAL_CKPT)}")
elif os.path.isdir(LOCAL_CKPT):
    import shutil
    for f in os.listdir(LOCAL_CKPT):
        shutil.move(os.path.join(LOCAL_CKPT, f), DRIVE_CKPT)
    shutil.rmtree(LOCAL_CKPT)
    os.symlink(DRIVE_CKPT, LOCAL_CKPT)
    print(f"기존 파일 Drive 이전 후 symlink 생성: {LOCAL_CKPT} → {DRIVE_CKPT}")
else:
    os.symlink(DRIVE_CKPT, LOCAL_CKPT)
    print(f"symlink 생성: {LOCAL_CKPT} → {DRIVE_CKPT}")

Mounted at /drive
symlink 생성: /content/checkpoints → /drive/MyDrive/aue8088-pa2/checkpoints


In [4]:
import math
import torch
from torch import nn
from torch.utils.data import DataLoader

from src.utils.seed import set_seed, seed_worker
from src.utils.transforms import train_transform, eval_transform, IMAGENET_MEAN, IMAGENET_STD
from src.utils.trainer import MultiTaskTrainer, TrainConfig
from src.utils.wandb_logger import WandbLogger
from src.utils.metrics import collect_predictions, confusion_matrices, per_class_prf, CLASS_NAMES
from src.datasets.bdd_attr import BDDAttrDataset, ATTRIBUTES
from src.datasets.samplers import multi_attr_balanced_sampler
from src.losses.imbalanced import FocalLoss
from src.augment.mix import cutmix_data, mixed_loss
from src.models.vit import vit_small_patch16_224

SEED = 42            # 전체 실험 고정 시드 (torch / numpy / random / cudnn.deterministic)
SKIP_TRAINING = False   # True: 학습 건너뛰고 Drive 체크포인트 로드 (평가 단계만 재현)
set_seed(SEED, deterministic=True)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}  |  SEED={SEED}  |  SKIP_TRAINING={SKIP_TRAINING}")


In [5]:
import wandb; wandb.login()   # API key 입력

WANDB_PROJECT = "aue8088-pa2"   # 비활성화하려면 None
WANDB_TAGS    = ["level3"]
# 각 실험마다 RUN_NAME 만 바꿔서 동일 프로젝트에 누적하세요.
EXPERIMENT_NAME = "ir-focal+ir-sampler+randaug+cutmix"

wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: qkrtjdgk16 (qkrtjdgk16-hanyang-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


In [6]:
DATA_ROOT = "../data/set_a"
BATCH = 64

# --- 데이터셋 자동 다운로드 (Google Drive) ---------------------------------
import os, sys, zipfile, subprocess
from torchvision import transforms

GDRIVE_FILE_ID = "1L7YC70QlO87aIbE5lbtQ94HUINJijBKK"
ZIP_PATH   = "../aue8088_pa2_data.zip"
EXTRACT_TO = ".."

if not os.path.isdir(DATA_ROOT):
    try:
        import gdown
    except ImportError:
        subprocess.run([sys.executable, "-m", "pip", "install", "-q", "gdown"], check=True)
        import gdown

    if not os.path.exists(ZIP_PATH):
        print("데이터셋 zip 다운로드 중...")
        gdown.download(id=GDRIVE_FILE_ID, output=ZIP_PATH, quiet=False)

    print("압축 해제 중...")
    with zipfile.ZipFile(ZIP_PATH) as zf:
        zf.extractall(EXTRACT_TO)
    print(f"완료 → {DATA_ROOT}")
else:
    print(f"데이터셋이 이미 존재합니다 → {DATA_ROOT}")
# --------------------------------------------------------------------------

# RandAugment 포함 train transform (level3 전용)
def train_transform_l3(img_size=224):
    return transforms.Compose([
        transforms.RandomResizedCrop(img_size, scale=(0.7, 1.0)),
        transforms.RandomHorizontalFlip(),
        transforms.RandAugment(num_ops=2, magnitude=9),
        transforms.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2),
        transforms.ToTensor(),
        transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
    ])

train_ds = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform_l3())
val_ds   = BDDAttrDataset(DATA_ROOT, "val",   transform=eval_transform())

# 속성별 IR (Imbalance Ratio) 자동 계산
IR = {}
for a in ATTRIBUTES:
    c = train_ds.class_counts(a).float()
    IR[a] = (c.max() / c[c > 0].min()).item()
print("Imbalance Ratio:", {a: round(v, 1) for a, v in IR.items()})

# 3속성 IR 통합 multi-attribute balanced sampler
sampler = multi_attr_balanced_sampler(train_ds, IR)
train_loader = DataLoader(train_ds, batch_size=BATCH, sampler=sampler, num_workers=2, pin_memory=True)
val_loader   = DataLoader(val_ds,   batch_size=BATCH, shuffle=False,   num_workers=2, pin_memory=True)

for a in ATTRIBUTES:
    print(f"{a:10s} 클래스별 샘플 수 (train) = {train_ds.class_counts(a).tolist()}")

데이터셋 zip 다운로드 중...


Downloading...
From (original): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK
From (redirected): https://drive.google.com/uc?id=1L7YC70QlO87aIbE5lbtQ94HUINJijBKK&confirm=t&uuid=e9a0320b-9b5c-4ed7-b210-68ecb5e2b936
To: /content/aue8088_pa2_data.zip
100%|██████████| 243M/243M [00:02<00:00, 97.6MB/s] 


압축 해제 중...
완료 → ../data/set_a
Imbalance Ratio: {'weather': 15.5, 'scene': 5.4, 'timeofday': 6.3}
weather    클래스별 샘플 수 (train) = [3100, 800, 400, 200, 0, 500]
scene      클래스별 샘플 수 (train) = [3052, 1386, 562]
timeofday  클래스별 샘플 수 (train) = [2479, 2129, 392]


In [8]:
# IR 기반 gamma 자동 계산 (불균형 클수록 gamma 높게)
gamma_dict = {a: round(1.0 + math.log(IR[a]), 1) for a in ATTRIBUTES}
print(f"FocalLoss gamma — {gamma_dict}")

# 속성별 FocalLoss (IR로 자동 결정된 gamma)
loss_fns = {a: FocalLoss(gamma=gamma_dict[a]).to(device) for a in ATTRIBUTES}

# IR 정규화 가중치 — mixed_loss 및 wandb 로깅용
ir_total    = sum(IR.values())
mix_weights = {a: IR[a] / ir_total for a in ATTRIBUTES}

set_seed(SEED, deterministic=True)
model  = vit_small_patch16_224().to(device)
epochs = 30
optim  = torch.optim.AdamW(model.parameters(), lr=5e-4, weight_decay=5e-2)
sched  = torch.optim.lr_scheduler.CosineAnnealingLR(optim, T_max=epochs)

logger = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level3-{EXPERIMENT_NAME}",
    config={
        "backbone": "vit_s16",
        "sampler": "multi_attr_ir_balanced",
        "loss": {a: f"focal_g{gamma_dict[a]}" for a in ATTRIBUTES},
        "augmentation": "randaugment(n=2,m=9)+cutmix(a=1.0)",
        "IR": {a: round(v, 1) for a, v in IR.items()},
        "mix_weights": {a: round(v, 3) for a, v in mix_weights.items()},
        "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
    },
    tags=WANDB_TAGS + [EXPERIMENT_NAME],
)
trainer = MultiTaskTrainer(model, optim, sched, loss_fns, device, TrainConfig(epochs=epochs), logger=logger)

FocalLoss gamma — {'weather': 3.7, 'scene': 2.7, 'timeofday': 2.8}


wandb: WARNING Using a boolean value for 'reinit' is deprecated. Use 'return_previous' or 'finish_previous' instead.


In [9]:
_ckpt_main = f"../checkpoints/level3_{EXPERIMENT_NAME}.pth"

if SKIP_TRAINING and os.path.exists(_ckpt_main):
    ckpt = torch.load(_ckpt_main, map_location=device)
    model.load_state_dict(ckpt["state_dict"])
    model.eval()
    history = ckpt.get("history", {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []})
    print(f"[SKIP] {EXPERIMENT_NAME} 체크포인트 로드 완료")
else:
    from tqdm import tqdm

    def step_with_mix(images, targets):
        """CutMix + IR 기반 속성별 가중 loss."""
        x, ya, yb, lam = cutmix_data(images, targets, alpha=1.0)
        logits = model(x)
        return mixed_loss(loss_fns, logits, ya, yb, lam, weights=mix_weights)

    history = {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []}

    for epoch in range(epochs):
        model.train()
        running, n_batches = 0.0, 0

        pbar = tqdm(train_loader, desc=f"train e{epoch+1:02d}", leave=False)
        for batch in pbar:
            images  = batch["image"].to(device, non_blocking=True)
            targets = {a: batch[a].to(device, non_blocking=True) for a in ATTRIBUTES}

            optim.zero_grad(set_to_none=True)
            loss = step_with_mix(images, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optim.step()

            running   += loss.item()
            n_batches += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        train_loss  = running / max(n_batches, 1)
        val_metrics = trainer.evaluate(val_loader)
        sched.step()

        history["train_loss"].append(train_loss)
        history["val_avg_mf1"].append(val_metrics["avg_macro_f1"])
        history["val_per_mf1"].append(val_metrics["per_macro_f1"])

        print(
            f"[epoch {epoch+1:02d}/{epochs}] "
            f"loss={train_loss:.4f}  "
            f"val_avg_MF1={val_metrics['avg_macro_f1']:.4f}  "
            f"per={val_metrics['per_macro_f1']}"
        )

        log_payload = {
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "val/avg_macro_f1": val_metrics["avg_macro_f1"],
            "lr": optim.param_groups[0]["lr"],
        }
        for a, v in val_metrics["per_macro_f1"].items():
            log_payload[f"val/mf1_{a}"] = v
        logger.log(log_payload, step=epoch)

    os.makedirs("../checkpoints", exist_ok=True)
    torch.save({"state_dict": model.state_dict(), "history": history}, _ckpt_main)
    print(f"저장 완료 → {_ckpt_main}")


[epoch 01/30] loss=0.5605  val_avg_MF1=0.3321  per={'weather': 0.12644444444444444, 'scene': 0.2531017369727047, 'timeofday': 0.6168044182383929}


[epoch 02/30] loss=0.4953  val_avg_MF1=0.3579  per={'weather': 0.1787787281732707, 'scene': 0.27147392290249434, 'timeofday': 0.6235908731503444}


[epoch 03/30] loss=0.4878  val_avg_MF1=0.3966  per={'weather': 0.2951536052801875, 'scene': 0.253416149068323, 'timeofday': 0.6412235671884572}


[epoch 04/30] loss=0.4705  val_avg_MF1=0.3964  per={'weather': 0.24158246294469202, 'scene': 0.32339371038184034, 'timeofday': 0.6242269895882525}


[epoch 05/30] loss=0.4725  val_avg_MF1=0.3683  per={'weather': 0.21232050050493856, 'scene': 0.2531017369727047, 'timeofday': 0.6396053677862455}


...

[epoch 26/30] loss=0.4137  val_avg_MF1=0.4919  per={'weather': 0.38940183643406784, 'scene': 0.34927259501727587, 'timeofday': 0.7369684967575747}


[epoch 27/30] loss=0.4084  val_avg_MF1=0.4947  per={'weather': 0.38954599125514083, 'scene': 0.3624700739044447, 'timeofday': 0.7320320690028241}


[epoch 28/30] loss=0.4070  val_avg_MF1=0.4937  per={'weather': 0.3952303667823897, 'scene': 0.35117477069983666, 'timeofday': 0.7346948407720428}


[epoch 29/30] loss=0.4155  val_avg_MF1=0.4908  per={'weather': 0.3922140354886585, 'scene': 0.3545203828593942, 'timeofday': 0.7256751067268472}


[epoch 30/30] loss=0.4098  val_avg_MF1=0.4958  per={'weather': 0.4013019339595956, 'scene': 0.35783539710241286, 'timeofday': 0.7281401608142201}
저장 완료 → ../checkpoints/level3_ir-focal+ir-sampler+randaug+cutmix.pth


In [10]:
# 학습 종료 후 — 속성별 confusion matrix + per-class F1 표를 wandb 에 업로드
val_pred, _, val_tgt, _ = collect_predictions(model, val_loader, device)
cms = confusion_matrices(val_pred, val_tgt)
prf = per_class_prf(val_pred, val_tgt)
for a in ATTRIBUTES:
    logger.log_confusion_matrix(f"final/cm_{a}", cms[a], CLASS_NAMES[a])
    rows = list(zip(prf[a]["class"], prf[a]["precision"], prf[a]["recall"], prf[a]["f1"], prf[a]["support"]))
    logger.log_table(f"final/prf_{a}", ["class", "P", "R", "F1", "support"], [list(r) for r in rows])
logger.finish()

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▅▅▄▄▄▃▃▃▃▃▂▂▂▂▂▂▂▂▂▁▁▁▁▁▁▁▁▁▁
val/avg_macro_f1,▁▂▄▄▃▄▄▄▅▆▆▅▆▆▅▆▇▅▆▇▆█▇███████
val/mf1_scene,▁▂▁▅▁▁▁▇▅▆▇▇▇█▁▇▇▆█▇█▆▇█▇▇█▇▇█
val/mf1_timeofday,▂▃▃▃▃▃▃▁▃▆█▄▆▅▆▅▅▄▅▆▆█▇█████▇▇
val/mf1_weather,▁▂▅▄▃▅▅▅▅▅▅▅▅▆▆▆█▅▆▇▅█▇▇█▇▇▇▇▇
epoch,30
lr,0
train/loss,0.40978
val/avg_macro_f1,0.49576


In [11]:
# ── Ablation B: IR sampler only (FocalLoss X, RandAugment X, CutMix X) ──────
EXPERIMENT_NAME_B = "ir-sampler-only"

# 기본 augmentation (RandAugment 없음)
train_ds_b     = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
sampler_b      = multi_attr_balanced_sampler(train_ds_b, IR)
train_loader_b = DataLoader(train_ds_b, batch_size=BATCH, sampler=sampler_b,
                            num_workers=2, pin_memory=True)

# CE loss (FocalLoss 없음)
loss_fns_b = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}

set_seed(SEED, deterministic=True)
model_b = vit_small_patch16_224().to(device)
optim_b = torch.optim.AdamW(model_b.parameters(), lr=5e-4, weight_decay=5e-2)
sched_b = torch.optim.lr_scheduler.CosineAnnealingLR(optim_b, T_max=epochs)

logger_b = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level3-{EXPERIMENT_NAME_B}",
    config={
        "backbone": "vit_s16",
        "sampler": "multi_attr_ir_balanced",
        "loss": "cross_entropy",
        "augmentation": "standard",
        "IR": {a: round(v, 1) for a, v in IR.items()},
        "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
    },
    tags=WANDB_TAGS + [EXPERIMENT_NAME_B],
)
trainer_b = MultiTaskTrainer(model_b, optim_b, sched_b, loss_fns_b, device,
                             TrainConfig(epochs=epochs), logger=logger_b)

In [12]:
_ckpt_b = f"../checkpoints/level3_{EXPERIMENT_NAME_B}.pth"

if SKIP_TRAINING and os.path.exists(_ckpt_b):
    ckpt = torch.load(_ckpt_b, map_location=device)
    model_b.load_state_dict(ckpt["state_dict"])
    model_b.eval()
    history_b = ckpt.get("history", {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []})
    print(f"[SKIP] {EXPERIMENT_NAME_B} 체크포인트 로드 완료")
else:
    from tqdm import tqdm

    history_b = {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []}

    for epoch in range(epochs):
        model_b.train()
        running, n_batches = 0.0, 0

        pbar = tqdm(train_loader_b, desc=f"train e{epoch+1:02d}", leave=False)
        for batch in pbar:
            images  = batch["image"].to(device, non_blocking=True)
            targets = {a: batch[a].to(device, non_blocking=True) for a in ATTRIBUTES}

            optim_b.zero_grad(set_to_none=True)
            logits = model_b(images)
            loss   = sum(loss_fns_b[a](logits[a], targets[a]) for a in ATTRIBUTES)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_b.parameters(), 1.0)
            optim_b.step()

            running   += loss.item()
            n_batches += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        train_loss  = running / max(n_batches, 1)
        val_metrics = trainer_b.evaluate(val_loader)
        sched_b.step()

        history_b["train_loss"].append(train_loss)
        history_b["val_avg_mf1"].append(val_metrics["avg_macro_f1"])
        history_b["val_per_mf1"].append(val_metrics["per_macro_f1"])

        print(
            f"[epoch {epoch+1:02d}/{epochs}] "
            f"loss={train_loss:.4f}  "
            f"val_avg_MF1={val_metrics['avg_macro_f1']:.4f}  "
            f"per={val_metrics['per_macro_f1']}"
        )

        log_payload = {
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "val/avg_macro_f1": val_metrics["avg_macro_f1"],
            "lr": optim_b.param_groups[0]["lr"],
        }
        for a, v in val_metrics["per_macro_f1"].items():
            log_payload[f"val/mf1_{a}"] = v
        logger_b.log(log_payload, step=epoch)

    torch.save({"state_dict": model_b.state_dict(), "history": history_b}, _ckpt_b)
    print(f"저장 완료 → {_ckpt_b}")


[epoch 01/30] loss=2.9031  val_avg_MF1=0.4364  per={'weather': 0.2918123568582284, 'scene': 0.35083769343277343, 'timeofday': 0.6665337083760586}


[epoch 02/30] loss=2.6207  val_avg_MF1=0.4189  per={'weather': 0.33374200946664717, 'scene': 0.2581109847490742, 'timeofday': 0.6648777918604379}


[epoch 03/30] loss=2.5618  val_avg_MF1=0.4791  per={'weather': 0.32369893558837143, 'scene': 0.37117182665127874, 'timeofday': 0.7424053634487997}


[epoch 04/30] loss=2.5275  val_avg_MF1=0.4860  per={'weather': 0.32500375544039467, 'scene': 0.3781965178902662, 'timeofday': 0.7548265280599328}


[epoch 05/30] loss=2.4685  val_avg_MF1=0.4814  per={'weather': 0.380619221550435, 'scene': 0.3140624347898669, 'timeofday': 0.7495130470407023}


...

[epoch 26/30] loss=1.8895  val_avg_MF1=0.5631  per={'weather': 0.4588026893846504, 'scene': 0.419831223628692, 'timeofday': 0.8105548198571454}


[epoch 27/30] loss=1.8939  val_avg_MF1=0.5766  per={'weather': 0.4446293624339106, 'scene': 0.44650672971718525, 'timeofday': 0.8387012135874427}


[epoch 28/30] loss=1.8952  val_avg_MF1=0.5746  per={'weather': 0.46838455826638487, 'scene': 0.4423278731306939, 'timeofday': 0.8132036154779761}


[epoch 29/30] loss=1.8663  val_avg_MF1=0.5708  per={'weather': 0.4561900262197243, 'scene': 0.4396309880457146, 'timeofday': 0.8165966271761453}


[epoch 30/30] loss=1.8696  val_avg_MF1=0.5738  per={'weather': 0.458747654161553, 'scene': 0.4493920519236974, 'timeofday': 0.8132036154779761}
저장 완료 → ../checkpoints/level3_ir-sampler-only.pth


In [1]:
val_pred_b, _, val_tgt_b, _ = collect_predictions(model_b, val_loader, device)
cms_b = confusion_matrices(val_pred_b, val_tgt_b)
prf_b = per_class_prf(val_pred_b, val_tgt_b)
for a in ATTRIBUTES:
    logger_b.log_confusion_matrix(f"final/cm_{a}", cms_b[a], CLASS_NAMES[a])
    rows = list(zip(prf_b[a]["class"], prf_b[a]["precision"], prf_b[a]["recall"],
                    prf_b[a]["f1"], prf_b[a]["support"]))
    logger_b.log_table(f"final/prf_{a}", ["class", "P", "R", "F1", "support"],
                       [list(r) for r in rows])
logger_b.finish()

NameError: name 'collect_predictions' is not defined

In [9]:
# ── Ablation C: CutMix only (sampler X, FocalLoss X, RandAugment X) ─────────
EXPERIMENT_NAME_C = "cutmix-only"

# 기본 DataLoader (sampler 없음, shuffle)
train_ds_c     = BDDAttrDataset(DATA_ROOT, "train", transform=train_transform())
train_loader_c = DataLoader(train_ds_c, batch_size=BATCH, shuffle=True,
                            num_workers=2, pin_memory=True)

# CE loss, 균등 mix_weights (1/3씩)
loss_fns_c    = {a: nn.CrossEntropyLoss() for a in ATTRIBUTES}
mix_weights_c = {a: 1.0 / len(ATTRIBUTES) for a in ATTRIBUTES}

set_seed(SEED, deterministic=True)
model_c = vit_small_patch16_224().to(device)
optim_c = torch.optim.AdamW(model_c.parameters(), lr=5e-4, weight_decay=5e-2)
sched_c = torch.optim.lr_scheduler.CosineAnnealingLR(optim_c, T_max=epochs)

logger_c = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level3-{EXPERIMENT_NAME_C}",
    config={
        "backbone": "vit_s16",
        "sampler": "none",
        "loss": "cross_entropy",
        "augmentation": "cutmix(a=1.0)",
        "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
    },
    tags=WANDB_TAGS + [EXPERIMENT_NAME_C],
)
trainer_c = MultiTaskTrainer(model_c, optim_c, sched_c, loss_fns_c, device,
                             TrainConfig(epochs=epochs), logger=logger_c)

In [10]:
_ckpt_c = f"../checkpoints/level3_{EXPERIMENT_NAME_C}.pth"

if SKIP_TRAINING and os.path.exists(_ckpt_c):
    ckpt = torch.load(_ckpt_c, map_location=device)
    model_c.load_state_dict(ckpt["state_dict"])
    model_c.eval()
    history_c = ckpt.get("history", {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []})
    print(f"[SKIP] {EXPERIMENT_NAME_C} 체크포인트 로드 완료")
else:
    from tqdm import tqdm

    def step_cutmix_only(images, targets):
        x, ya, yb, lam = cutmix_data(images, targets, alpha=1.0)
        logits = model_c(x)
        return mixed_loss(loss_fns_c, logits, ya, yb, lam, weights=mix_weights_c)

    history_c = {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []}

    for epoch in range(epochs):
        model_c.train()
        running, n_batches = 0.0, 0

        pbar = tqdm(train_loader_c, desc=f"train e{epoch+1:02d}", leave=False)
        for batch in pbar:
            images  = batch["image"].to(device, non_blocking=True)
            targets = {a: batch[a].to(device, non_blocking=True) for a in ATTRIBUTES}

            optim_c.zero_grad(set_to_none=True)
            loss = step_cutmix_only(images, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_c.parameters(), 1.0)
            optim_c.step()

            running   += loss.item()
            n_batches += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        train_loss  = running / max(n_batches, 1)
        val_metrics = trainer_c.evaluate(val_loader)
        sched_c.step()

        history_c["train_loss"].append(train_loss)
        history_c["val_avg_mf1"].append(val_metrics["avg_macro_f1"])
        history_c["val_per_mf1"].append(val_metrics["per_macro_f1"])

        print(
            f"[epoch {epoch+1:02d}/{epochs}] "
            f"loss={train_loss:.4f}  "
            f"val_avg_MF1={val_metrics['avg_macro_f1']:.4f}  "
            f"per={val_metrics['per_macro_f1']}"
        )

        log_payload = {
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "val/avg_macro_f1": val_metrics["avg_macro_f1"],
            "lr": optim_c.param_groups[0]["lr"],
        }
        for a, v in val_metrics["per_macro_f1"].items():
            log_payload[f"val/mf1_{a}"] = v
        logger_c.log(log_payload, step=epoch)

    torch.save({"state_dict": model_c.state_dict(), "history": history_c}, _ckpt_c)
    print(f"저장 완료 → {_ckpt_c}")


[epoch 01/30] loss=0.9246  val_avg_MF1=0.3355  per={'weather': 0.14975410819239535, 'scene': 0.2531017369727047, 'timeofday': 0.6036970700550843}


[epoch 02/30] loss=0.8498  val_avg_MF1=0.3794  per={'weather': 0.20471500689299493, 'scene': 0.3132721171221005, 'timeofday': 0.6201135852451642}


[epoch 03/30] loss=0.8368  val_avg_MF1=0.3728  per={'weather': 0.22819035587356562, 'scene': 0.2531017369727047, 'timeofday': 0.6371309443317558}


[epoch 04/30] loss=0.8336  val_avg_MF1=0.3599  per={'weather': 0.1380491307813104, 'scene': 0.30331052921286467, 'timeofday': 0.6384801177373584}


[epoch 05/30] loss=0.8279  val_avg_MF1=0.4154  per={'weather': 0.2224052869923788, 'scene': 0.3498720363127143, 'timeofday': 0.6738673524975692}


...

[epoch 26/30] loss=0.7649  val_avg_MF1=0.4775  per={'weather': 0.31920719063846664, 'scene': 0.36319582113974636, 'timeofday': 0.7501435046524572}


[epoch 27/30] loss=0.7575  val_avg_MF1=0.4830  per={'weather': 0.3293132646713209, 'scene': 0.3696794296946475, 'timeofday': 0.7500988119318315}


[epoch 28/30] loss=0.7612  val_avg_MF1=0.4762  per={'weather': 0.31318770482909347, 'scene': 0.36539036386858337, 'timeofday': 0.7500988119318315}


[epoch 29/30] loss=0.7679  val_avg_MF1=0.4710  per={'weather': 0.3105328013803791, 'scene': 0.3632799843010546, 'timeofday': 0.7391135772558929}


[epoch 30/30] loss=0.7661  val_avg_MF1=0.4706  per={'weather': 0.3105328013803791, 'scene': 0.3621885797417712, 'timeofday': 0.7390664073353709}
저장 완료 → ../checkpoints/level3_cutmix-only.pth


In [11]:
val_pred_c, _, val_tgt_c, _ = collect_predictions(model_c, val_loader, device)
cms_c = confusion_matrices(val_pred_c, val_tgt_c)
prf_c = per_class_prf(val_pred_c, val_tgt_c)
for a in ATTRIBUTES:
    logger_c.log_confusion_matrix(f"final/cm_{a}", cms_c[a], CLASS_NAMES[a])
    rows = list(zip(prf_c[a]["class"], prf_c[a]["precision"], prf_c[a]["recall"],
                    prf_c[a]["f1"], prf_c[a]["support"]))
    logger_c.log_table(f"final/prf_{a}", ["class", "P", "R", "F1", "support"],
                       [list(r) for r in rows])
logger_c.finish()

epoch,▁▁▁▂▂▂▂▃▃▃▃▄▄▄▄▅▅▅▅▆▆▆▆▇▇▇▇███
lr,█████▇▇▇▇▆▆▆▅▅▅▄▄▃▃▃▂▂▂▂▁▁▁▁▁▁
train/loss,█▅▄▄▄▄▃▃▃▃▃▂▂▃▂▂▃▂▂▂▂▂▁▁▁▁▁▁▁▁
val/avg_macro_f1,▁▃▃▂▅▃▃▅▅▆▄▅▆▅▅▆▇▆▇█▇▇▇▇█▇█▇▇▇
val/mf1_scene,▁▄▁▃▅▂▂▄▆▇▃▆▃▄▆▆▆▆██▆▆▆▇█▆▆▆▆▆
val/mf1_timeofday,▁▂▂▂▄▂▃▇▅▆▅▅▇▄▄▆▆▇▅▇▆█▇▇▇▇▇▇▇▇
val/mf1_weather,▁▃▄▁▄▄▄▃▄▄▄▄▇▄▅▆▆▅▆▇▇▆▅▇███▇▇▇
epoch,30
lr,0
train/loss,0.76607
val/avg_macro_f1,0.4706


In [12]:
# ── Pretrained ViT + IR전략 (Level5 베이스 모델용) ───────────────────────────
# DeiT-S/16 ImageNet pretrained → 동일 IR 전략 적용
# 체크포인트: level3_vit-pretrained+ir-focal+ir-sampler+randaug+cutmix.pth
import subprocess

EXPERIMENT_NAME_P = "vit-pretrained+ir-focal+ir-sampler+randaug+cutmix"
DEIT_CKPT = "../deit_s16.pth"

if not os.path.exists(DEIT_CKPT):
    print("DeiT-S/16 체크포인트 다운로드 중...")
    subprocess.run([
        "wget", "-q",
        "https://dl.fbaipublicfiles.com/deit/deit_small_patch16_224-cd65a155.pth",
        "-O", DEIT_CKPT,
    ], check=True)
    print("완료")


def remap_deit(state_dict):
    new_sd = {}
    for k, v in state_dict.items():
        if k.startswith("head."):
            continue
        k = k.replace(".mlp.fc1.", ".mlp.0.")
        k = k.replace(".mlp.fc2.", ".mlp.3.")
        new_sd[k] = v
    return new_sd


set_seed(SEED, deterministic=True)
model_p = vit_small_patch16_224().to(device)

raw = torch.load(DEIT_CKPT, map_location="cpu")
sd  = raw.get("model", raw)
missing, unexpected = model_p.load_state_dict(remap_deit(sd), strict=False)
print(f"pretrained 로드 완료 — missing: {len(missing)}, unexpected: {len(unexpected)}")

optim_p = torch.optim.AdamW(model_p.parameters(), lr=5e-4, weight_decay=5e-2)
sched_p = torch.optim.lr_scheduler.CosineAnnealingLR(optim_p, T_max=epochs)

logger_p = WandbLogger(
    project=WANDB_PROJECT,
    run_name=f"level3-{EXPERIMENT_NAME_P}",
    config={
        "backbone": "vit_s16", "pretrained": True,
        "sampler": "multi_attr_ir_balanced",
        "loss": {a: f"focal_g{gamma_dict[a]}" for a in ATTRIBUTES},
        "augmentation": "randaugment(n=2,m=9)+cutmix(a=1.0)",
        "IR": {a: round(v, 1) for a, v in IR.items()},
        "mix_weights": {a: round(v, 3) for a, v in mix_weights.items()},
        "epochs": epochs, "batch": BATCH, "lr": 5e-4, "weight_decay": 5e-2, "seed": SEED,
    },
    tags=WANDB_TAGS + [EXPERIMENT_NAME_P, "pretrained"],
)
trainer_p = MultiTaskTrainer(model_p, optim_p, sched_p, loss_fns, device,
                             TrainConfig(epochs=epochs), logger=logger_p)

DeiT-S/16 체크포인트 다운로드 중...
완료
pretrained 로드 완료 — missing: 6, unexpected: 0


In [13]:
_ckpt_p = f"../checkpoints/level3_{EXPERIMENT_NAME_P}.pth"

if SKIP_TRAINING and os.path.exists(_ckpt_p):
    ckpt = torch.load(_ckpt_p, map_location=device)
    model_p.load_state_dict(ckpt["state_dict"])
    model_p.eval()
    history_p = ckpt.get("history", {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []})
    print(f"[SKIP] {EXPERIMENT_NAME_P} 체크포인트 로드 완료")
else:
    from tqdm import tqdm

    def step_pretrained_mix(images, targets):
        """CutMix + IR 기반 속성별 가중 loss (pretrained 모델)."""
        x, ya, yb, lam = cutmix_data(images, targets, alpha=1.0)
        logits = model_p(x)
        return mixed_loss(loss_fns, logits, ya, yb, lam, weights=mix_weights)

    history_p = {"train_loss": [], "val_avg_mf1": [], "val_per_mf1": []}

    for epoch in range(epochs):
        model_p.train()
        running, n_batches = 0.0, 0

        pbar = tqdm(train_loader, desc=f"train e{epoch+1:02d}", leave=False)
        for batch in pbar:
            images  = batch["image"].to(device, non_blocking=True)
            targets = {a: batch[a].to(device, non_blocking=True) for a in ATTRIBUTES}

            optim_p.zero_grad(set_to_none=True)
            loss = step_pretrained_mix(images, targets)
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model_p.parameters(), 1.0)
            optim_p.step()

            running   += loss.item()
            n_batches += 1
            pbar.set_postfix({"loss": f"{loss.item():.4f}"})

        train_loss  = running / max(n_batches, 1)
        val_metrics = trainer_p.evaluate(val_loader)
        sched_p.step()

        history_p["train_loss"].append(train_loss)
        history_p["val_avg_mf1"].append(val_metrics["avg_macro_f1"])
        history_p["val_per_mf1"].append(val_metrics["per_macro_f1"])

        print(
            f"[epoch {epoch+1:02d}/{epochs}] "
            f"loss={train_loss:.4f}  "
            f"val_avg_MF1={val_metrics['avg_macro_f1']:.4f}  "
            f"per={val_metrics['per_macro_f1']}"
        )

        log_payload = {
            "epoch": epoch + 1,
            "train/loss": train_loss,
            "val/avg_macro_f1": val_metrics["avg_macro_f1"],
            "lr": optim_p.param_groups[0]["lr"],
        }
        for a, v in val_metrics["per_macro_f1"].items():
            log_payload[f"val/mf1_{a}"] = v
        logger_p.log(log_payload, step=epoch)

    os.makedirs("../checkpoints", exist_ok=True)
    torch.save({"state_dict": model_p.state_dict(), "history": history_p}, _ckpt_p)
    print(f"저장 완료 → {_ckpt_p}")


[epoch 01/30] loss=0.5060  val_avg_MF1=0.4331  per={'weather': 0.2972535294682946, 'scene': 0.37299368333851096, 'timeofday': 0.6290557474768002}


[epoch 02/30] loss=0.4326  val_avg_MF1=0.4277  per={'weather': 0.33545099484200197, 'scene': 0.3082962727657772, 'timeofday': 0.639462442275421}


[epoch 03/30] loss=0.4126  val_avg_MF1=0.5263  per={'weather': 0.4505855172503394, 'scene': 0.35129921022487487, 'timeofday': 0.7769299152752603}


[epoch 04/30] loss=0.3877  val_avg_MF1=0.5337  per={'weather': 0.4644877403276286, 'scene': 0.39417706476530007, 'timeofday': 0.7423512849841227}


[epoch 05/30] loss=0.3731  val_avg_MF1=0.5916  per={'weather': 0.5061823973041555, 'scene': 0.4500910853852031, 'timeofday': 0.8184595538784297}


...

[epoch 26/30] loss=0.2152  val_avg_MF1=0.6911  per={'weather': 0.5611866729135478, 'scene': 0.6510999391257325, 'timeofday': 0.8608738303259392}


[epoch 27/30] loss=0.2076  val_avg_MF1=0.6910  per={'weather': 0.5585931129778338, 'scene': 0.6387373273261775, 'timeofday': 0.8755862664753041}


[epoch 28/30] loss=0.2006  val_avg_MF1=0.6887  per={'weather': 0.5677170459512236, 'scene': 0.641710821406139, 'timeofday': 0.856779127305443}


[epoch 29/30] loss=0.2119  val_avg_MF1=0.6979  per={'weather': 0.5757689844250725, 'scene': 0.6519623555658592, 'timeofday': 0.8658999281640791}


[epoch 30/30] loss=0.2101  val_avg_MF1=0.6991  per={'weather': 0.5780042345541561, 'scene': 0.6534407314806588, 'timeofday': 0.8658999281640791}
저장 완료 → ../checkpoints/level3_vit-pretrained+ir-focal+ir-sampler+randaug+cutmix.pth


In [ ]:
val_pred_p, _, val_tgt_p, _ = collect_predictions(model_p, val_loader, device)
cms_p = confusion_matrices(val_pred_p, val_tgt_p)
prf_p = per_class_prf(val_pred_p, val_tgt_p)
for a in ATTRIBUTES:
    logger_p.log_confusion_matrix(f"final/cm_{a}", cms_p[a], CLASS_NAMES[a])
    rows = list(zip(prf_p[a]["class"], prf_p[a]["precision"], prf_p[a]["recall"],
                    prf_p[a]["f1"], prf_p[a]["support"]))
    logger_p.log_table(f"final/prf_{a}", ["class", "P", "R", "F1", "support"],
                       [list(r) for r in rows])
logger_p.finish()
print("✓ Pretrained ViT + IR전략 실험 완료")

## 분석 (필수)

각 기법에 대해 **속성별 per-class F1 표** 를 작성하세요. 다음 항목을 강조해 주세요.
- 소수 클래스 (foggy / snowy / dawn-dusk) 의 적용 전후 성능 차이.
- 다수 클래스의 회귀 (regression) 발생 여부 — 그 trade-off 가 정당한지 논거.
- Sampling 과 Mixup / CutMix 의 상호작용 — 서로 도움이 되는지 충돌하는지.